# A-Maze-ing

## Read the configuration file

Just open the file, split the options turn them into a dictionary.

In [1]:
with open("config.txt") as f:
    txt = f.read()
    opts = txt.split("\n")
    if opts[-1] == "":
        opts.pop()
    tup = [tuple(i.split("=")) for i in opts]
    conf = {i[0].upper(): i[1] for i in tup if i[0][0] != "#"}
print(conf)
try:
    size = (int(conf["HEIGHT"]), int(conf["WIDTH"]))
    start = conf["ENTRY"].split(",")
    end = conf["EXIT"].split(",")
    entr = (int(start[0]), int(start[1]))
    exit = (int(end[0]), int(end[1]))
except ValueError:
    print("[ERROR] Configurtion is in wrong format. Please, use integers")
except IndexError:
    print("[ERROR] Entry or exit in wrong format. Usage ex.: EXIT=int,int")

{'HEIGHT': '13', 'WIDTH': '19', 'ENTRY': '0,0', 'EXIT': '12,18', 'OUTPUT_FILE': 'maze.txt', 'PERFECT': 'False', 'SEED': '42', 'ALGORITHM': 'Wilson'}


## Create a $h \times w$ array

Two different arrays:

* The maze itself.
* The visited cells.

The 42 was hardcoded.

In [2]:
from numpy import zeros, ones


size = (int(conf["HEIGHT"]), int(conf["WIDTH"]))
maze = ones(size, int) * 15
visited = zeros(size, int)
perfect = conf["PERFECT"].lower().capitalize() != "False"

if visited.shape[0] < 6 or visited.shape[1] < 8:
    is_42 = 0
    print("The '42' cannot be printed in a maze this size.")
else:
    is_42 = 1
    hs, ws = (visited.shape[1] - 1) // 2 - 3, (visited.shape[0] - 1) // 2 - 2
    pos_42 = (
        (ws, hs), (ws + 1, hs), (ws + 2, hs), (ws + 2, hs + 1),
        (ws + 2, hs + 2), (ws + 3, hs + 2), (ws + 4, hs + 2),
        (ws, hs + 4), (ws, hs + 5), (ws, hs + 6), (ws + 1, hs + 6),
        (ws + 2, hs + 6), (ws + 2, hs + 5), (ws + 2, hs + 4),
        (ws + 3, hs + 4), (ws + 4, hs + 4), (ws + 4, hs + 5),
        (ws + 4, hs + 6)
    )
    for i in pos_42:
        visited[i[0]][i[1]] = -1
print(pos_42)
print(visited)
for i in maze:
    print(i)

((4, 6), (5, 6), (6, 6), (6, 7), (6, 8), (7, 8), (8, 8), (4, 10), (4, 11), (4, 12), (5, 12), (6, 12), (6, 11), (6, 10), (7, 10), (8, 10), (8, 11), (8, 12))
[[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 -1  0  0  0 -1 -1 -1  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 -1  0  0  0  0  0 -1  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 -1 -1 -1  0 -1 -1 -1  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0 -1  0 -1  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0 -1  0 -1 -1 -1  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]]
[15 15 15 15 15 15 15 15 15 15 15 15 15 15 15 15 15 15 15]
[15 

## Manage errors

Checks:

* Maze dimensions must be positive.
* Maze entrance and exit are not in the same tile.
* Maze entrance and exit are inside the maze.
* Maze entrance and exit are not inside forbidden tiles.
* All maze walls in a tile are coherent with the neighbouring tiles.

In [3]:
from numpy.typing import NDArray


class MazeError(Exception):
    def __init__(self, msg: str):
        self.msg = msg


start = conf["ENTRY"].split(",")
end = conf["EXIT"].split(",")
entr = (int(start[0]), int(start[1]))
exit = (int(end[0]), int(end[1]))

if entr == exit:
    raise MazeError("Maze entrance and exit must be different")

if size[0] <= 0 or size[1] <= 0:
    raise MazeError("Maze dimensions must be positive")

in_bounds = [
    entr[0] >= 0,
    entr[1] >= 0,
    entr[0] < size[1],
    entr[1] < size[0],
    exit[0] >= 0,
    exit[1] >= 0,
    exit[0] < size[1],
    exit[1] < size[0]
]
if False in in_bounds:
    raise MazeError("Entrance or exit of the maze seem to be out of bounds")

if visited.shape[0] > 5 or visited.shape[1] > 7:
    if True in [entr in pos_42, exit in pos_42]:
        raise MazeError("Entrance or exit of the maze seem to be inside the 42")


def get_walls(n: int) -> tuple[int]:
    if n > 15 or n < 0:
        print(n)
        raise ValueError
    bits = []
    bits.append(n // 8)
    n %= 8
    bits.append(n // 4)
    n %= 4
    bits.append(n // 2)
    n %= 2
    bits.append(n)
    bits.reverse()
    return tuple(bits)


def check_tile(maze: NDArray, coords: tuple[int]):
    i, j = coords
    tile = int(maze[j, i])
    msg = f"Maze tile {coords} has incoherent walls"
    walls = get_walls(tile)
    if j == 0:
        if walls[0] == 0:
            raise MazeError(f"{msg} with north border")
    else:
        if walls[0] != get_walls(maze[j - 1][i])[2]:
            raise MazeError(f"{msg} with tile {(j - 1, i)}")
    if j == maze.shape[0] - 1:
        if walls[2] == 0:
            raise MazeError(f"{msg} with south border")
    else:
        if walls[2] != get_walls(maze[j + 1][i])[0]:
            raise MazeError(f"{msg} with tile {(j + 1, i)}")
    if i == 0:
        if walls[3] == 0:
            raise MazeError(f"{msg} with west border")
    else:
        if walls[3] != get_walls(maze[j][i - 1])[1]:
            raise MazeError(f"{msg} with tile {(j, i - 1)}")
    if i == maze.shape[1] - 1:
        if walls[1] == 0:
            raise MazeError(f"{msg} with east border")
    else:
        if walls[1] != get_walls(maze[j][i + 1])[3]:
            raise MazeError(f"{msg} with tile {(j, i + 1)}")

MazeError: Entrance or exit of the maze seem to be out of bounds

## Maze generator

Initial thought: Wilson algorithm.

* First tile in the maze.
* Random tile makes random walk.
* Looping random walk erases loop.
* Random walk finds maze: becomes part of the maze.
* Repeat until all tiles are part of the maze.

Second algorithm: Aldous-Broder:

* First tile in the maze.
* Point in there makes random walk.
* If unvisited cell: connects both tiles, continue random walk.
* Else: continue random walk.
* Repeat until all tiles are visited.

Complementary algorithm: DFS:

* First tile in the maze.
* Random walk until reaches a tile with no neighbours to visit.
* Bactrack to a tile with neighbours to visit, repeat.
* Repeat until all tiles are visited.

Complementary algorithm: Prim:

* First tile in the maze goes to a set.
* Select a yet to visit neighbour to a tile in the set.
* Connect both tiles.
* If tile has all neighbouring tiles visited, remove it from the set.
* Repeat until set is empty.

To do if I feel like it:

* Hunt & Kill (may be easy after DFS)
* Eller (doable)
* Krunskal (pain)
* Origin Shift (2 painful 4 me)

In [ ]:
import random as rng


def manhattan(curr: tuple[int], neigh: tuple[int]) -> int:
    if curr == neigh:
        return 0
    return abs(curr[0] - neigh[0]) + abs(curr[1] - neigh[1])


def connect(maze: NDArray, curr: tuple[int], neigh: tuple[int]) -> None:
    if manhattan(curr, neigh) != 1:
        return
    if curr[1] == neigh[1] + 1 and get_walls(maze[curr[1], curr[0]])[0] == 1:
        maze[curr[1]][curr[0]] -= 1
        maze[neigh[1]][neigh[0]] -= 4
    if curr[1] == neigh[1] - 1 and get_walls(maze[curr[1], curr[0]])[2] == 1:
        maze[curr[1]][curr[0]] -= 4
        maze[neigh[1]][neigh[0]] -= 1
    if curr[0] == neigh[0] - 1 and get_walls(maze[curr[1], curr[0]])[1] == 1:
        maze[curr[1]][curr[0]] -= 2
        maze[neigh[1]][neigh[0]] -= 8
    if curr[0] == neigh[0] + 1 and get_walls(maze[curr[1], curr[0]])[3] == 1:
        maze[curr[1]][curr[0]] -= 8
        maze[neigh[1]][neigh[0]] -= 2


def direction() -> tuple[str, int]:
    if rng.random() < .5:
        axis = "V"
    else:
        axis = "H"
    if rng.random() < .5:
        step = -1
    else:
        step = 1
    return (axis, step)


def count_visited(visited: NDArray) -> int:
    n = 0
    for i in range(visited.shape[0]):
        for j in range(visited.shape[1]):
            if visited[i][j] == 1:
                n += 1
    return n


def wilson_erase(visited: NDArray, path: list[tuple], point: tuple[int]):
    while path[-1] != point:
        visited[path[-1][1], path[-1][0]] = 0
        path.pop()


try:
    rng.seed(conf["SEED"])
except KeyError:
    pass

# Initialization
v = -1
while v == -1:
    current = (rng.randrange(0, size[1] - 1), rng.randrange(0, size[0] - 1))
    v = visited[current[1], current[0]]
visited[current[1]][current[0]] = 1

# Recursive backtracker
"""lst = [current]
while len(lst) > 0:
    nvis = set()
    h, w = current[:]
    neighs = ((h - 1, w), (h, w + 1), (h + 1, w), (h, w - 1))
    for i in neighs:
        try:
            if True in [i[j] in (-1, maze.shape[j]) for j in [0, 1]]:
                raise IndexError
            if visited[i[:]] == 0:
                nvis.add(i[:])
        except IndexError:
            continue
    if len(nvis) == 0:
        lst.pop()
    else:
        new = rng.choice([*nvis])
        nvis.clear()
        connect(maze, current, new)
        visited[new[:]] = 1
        lst.append(new)
    try:
        current = lst[-1]
    except IndexError:
        pass"""

# Prim Algorithm
"""opened = set()
opened.add(tuple(current))
while len(opened) > 0:
    nvis = set()
    h, w = current[:]
    neighs = ((h - 1, w), (h, w + 1), (h + 1, w), (h, w - 1))
    for i in neighs:
        try:
            if True in [i[j] in (-1, maze.shape[j]) for j in [0, 1]]:
                raise IndexError
            if visited[i[:]] == 0:
                nvis.add(i[:])
        except IndexError:
            continue
    if len(nvis) == 0:
        opened.remove(current)
    else:
        new = rng.choice([*nvis])
        nvis.clear()
        connect(maze, current, new)
        visited[new[:]] = 1
        opened.add(new)
    if len(opened) > 0:
        current = rng.choice([*opened])"""

# Alodus-Broder Algorithm
"""yet = set()
for i in range(maze.shape[0]):
    for j in range(maze.shape[1]):
        if visited[i, j] == 0:
            yet.add((i, j))
while len(yet) > 0:
    direct = direction()
    try:
        if direct[0] == "V":
            neighbour = (current[0] + direct[1], current[1])
            if current[0] + direct[1] in (-1, maze.shape[0]):
                raise IndexError
        else:
            neighbour = (current[0], current[1] + direct[1])
            if current[1] + direct[1] in (-1, maze.shape[1]):
                raise IndexError
    except IndexError:
        continue
    if visited[neighbour[0], neighbour[1]] == -1:
        continue
    elif visited[neighbour[0], neighbour[1]] == 0:
        connect(maze, tuple(current), tuple(neighbour))
        yet.discard(neighbour)
    current = neighbour
    visited[current[0], current[1]] = 1"""

# Wilson Algorithm
yet = set()
for j in range(maze.shape[0]):
    for i in range(maze.shape[1]):
        if visited[j, i] == 0:
            yet.add((i, j))
while len(yet) > 0:
    path = []
    current = rng.choices([*yet], k=1)[0]
    visited[current[1], current[0]] = 2
    path.append(current)
    last_direction = ()
    while current in yet:
        direct = direction()
        while last_direction == direct:
            direct = direction()
        try:
            if direct[0] == "V":
                if current[1] + direct[1] in (-1, maze.shape[0]):
                    raise IndexError
                neighbour = (current[0], current[1] + direct[1])
            else:
                if current[0] + direct[1]  in (-1, maze.shape[1]):
                    raise IndexError
                neighbour = (current[0] + direct[1], current[1])
        except IndexError:
            continue
        if visited[neighbour[1], neighbour[0]] == -1:
            continue
        elif visited[neighbour[1], neighbour[0]] == 1:
            path.append(neighbour)
            break
        elif neighbour in path:
            wilson_erase(visited, path, neighbour)
        current = neighbour
        visited[current[1], current[0]] = 2
        if current not in path:
            path.append(current)
    for tile in path:
        visited[tile[1], tile[0]] = 1
        yet.discard(tile)
    while len(path) > 1:
        connect(maze, path[-2], path[-1])
        current = path[-1]
        path.pop()
    path.clear()

print(maze)
base = "0123456789abcdef"
for j in range(maze.shape[0]):
    for i in range(maze.shape[1]):
        try:
            check_tile(maze, (i, j))
            print(base[maze[j][i]], end="")
        except MazeError as e:
            print(e.msg)
    print()
print(maze)
if 0 in visited:
    print("There are still unvisited tiles")

In [ ]:
def break_wall(maze: NDArray, tile: tuple[int], op: str) -> None:
    i, j = tile[:]
    if op == "N":
        print(tile, (i - 1, j))
        connect(maze, tile, (i - 1, j))
    elif op == "E":
        print(tile, (i, j + 1))
        connect(maze, tile, (i, j + 1))
    elif op == "S":
        print(tile, (i + 1, j))
        connect(maze, tile, (i + 1, j))
    elif op == "W":
        print(tile, (i, j - 1))
        connect(maze, tile, (i, j - 1))


def remove_walls(maze: NDArray, tile: tuple[int], prob: float = 1) -> None:
    if maze[tile[1], tile[0]] == 15 or prob < 0 or prob > 1:
        return
    i, j = tile[:]
    walls = get_walls(maze[j, i])
    r_walls = []
    if j > 0:
        if maze[j - 1][i] != 15 and walls[0] == 1:
            r_walls.append("N")
    if i < maze.shape[1] - 1:
        if maze[j][i + 1] != 15 and walls[1] == 1:
            r_walls.append("E")
    if j < maze.shape[0] - 1:
        if maze[j + 1][i] != 15 and walls[2] == 1:
            r_walls.append("S")
    if i > 0:
        if maze[j][i - 1] != 15 and walls[3] == 1:
            r_walls.append("W")
    if len(r_walls) > 0 and rng.random() < prob:
        break_wall(maze, tile, rng.choice(r_walls))


if perfect is False:
    #print(maze)
    connect(maze, (0, 0), (0, 1))
    connect(maze, (0, 0), (1, 0))
    connect(maze, (0, maze.shape[1] - 1), (0, maze.shape[0] - 2))
    connect(maze, (0, maze.shape[1] - 1), (1, maze.shape[0] - 1))
    connect(maze, (maze.shape[1] - 1, maze.shape[0] - 1), (maze.shape[1] - 1, maze.shape[0] - 2))
    connect(maze, (maze.shape[1] - 1, maze.shape[0] - 1), (maze.shape[1] - 2, maze.shape[0] - 1))
    connect(maze, (maze.shape[1] - 1, 0), (maze.shape[1] - 1, 1))
    connect(maze, (maze.shape[1] - 1, 0), (maze.shape[1] - 2, 0))

    for _  in range(maze.shape[0] * maze.shape[1] // 3):
        tile = (rng.randrange(maze.shape[1]), rng.randrange(maze.shape[0]))
        remove_walls(maze, tile, .5)
    dead_ends = (7, 11, 13, 14)
    s_wall = ((0, 1), (0, 2), (0, 4), (0, 8))
    adm_walls = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]
    for j in range(maze.shape[0]):
        for i in range(maze.shape[1]):
            if maze[j, i] in dead_ends:
                remove_walls(maze, (i, j))
    for j in range(maze.shape[0]):
        for i in range(maze.shape[1]):
            if maze[j, i] == 0:
                adj_tiles = [(i - 1, j), (i, j + 1), (i + 1, j), (i, j - 1)]
                adj_walls = [maze[adj_tiles[k][1], adj_tiles[k][0]] in s_wall[k] for k in range(len(adj_tiles))]
                if False not in adj_walls:
                    maze[j, i] = rng.choice(adm_walls)
                    walls = get_walls(maze[i, j])
                    if walls[0] == 1:
                        maze[j][i - 1] += 4
                    if walls[1] == 1:
                        maze[j + 1][i] += 8
                    if walls[2] == 1:
                        maze[j][i + 1] += 1
                    if walls[3] == 1:
                        maze[j - 1][i] += 2

for j in range(maze.shape[0]):
    for i in range(maze.shape[1]):
        try:
            check_tile(maze, (i, j))
            print(base[maze[j][i]], end="")
        except MazeError as e:
            print(e.msg)
    print()

## Maze solution

A\* algorithm:

```python
# G cost: distance from starting node
# H (heuristic) cost: distance from end node
# h_cost = abs(x_1 - x_2) + abs(y_1 - y_2)
# F cost: G cost + H cost
# open: set of nodes to be evaluated
# closed: set of already evaluated nodes

while True:
    current = "node in open with lowest f_cost"
    remove(current, open)
    add(current, closed)
    
    if current == exit:
        return
    
    for i in neighbours(current):
        if accessible(i) == 0 or i in closed:
            continue

        if f_cost(i) < f_cost(current) or i not in open:
            set_f_cost(i)
            current = parent(i)
            if i not in open:
                add(i, open)
```

In [ ]:
from numpy import Inf


class Node():
    def __init__(self, value):
        self.val = value
        self._g = Inf
        self._h = Inf
        self._f = Inf
        self.next = None

    def add_front(self, newnode) -> None:
        if newnode is None:
            return
        newnode.next = self
        self = newnode

    def size(self) -> int:
        first = self
        n = 1
        while self.next != None and self.next is not first:
            n += 1
            self = self.next
        return n

    def get_g(self) -> int:
        return self._g

    def get_h(self) -> int:
        return self._h

    def get_f(self) -> int:
        return self._f

    def _set_f(self, f: int) -> None:
        self._f = f

    def set_g(self, g: int) -> None:
        self._g = g
        self._set_f(self.get_g() + self.get_h())

    def set_h(self, h: int) -> None:
        self._h = h
        self._set_f(self.get_g() + self.get_h())


def neighbours(maze: NDArray, current: tuple[int]) -> list[tuple]:
    n = maze[current[1], current[0]]
    neighs = []
    walls = get_walls(n)
    if walls[0] == 0:
        neighs.append((current[0], current[1] - 1))
    if walls[1] == 0:
        neighs.append((current[0] + 1, current[1]))
    if walls[2] == 0:
        neighs.append((current[0], current[1] + 1))
    if walls[3] == 0:
        neighs.append((current[0] - 1, current[1]))
    return neighs


def choose_current(f: NDArray, points: list[Node], exit: tuple[int]) -> Node:
    n = Inf
    f_cost = []
    for i in points:
        p = i.val
        if p == exit:
            return i
        f_cost.append(f[p[1], p[0]])
    min_f = min(f_cost)
    to_select = []
    for j in range(len(f_cost)):
        if f_cost[j] == min_f:
            to_select.append(points[j])
    min_h = Inf
    for k in to_select:
        h = manhattan(k.val, exit)
        if h < min_h:
            min_h = h
            current = k
    return current


def check_closed(closed: set[Node], point: tuple[int]) -> bool:
    for i in closed:
        if i.val == point:
            return True
    return False


def check_node(set: set[Node], point = tuple[int]) -> bool:
    for i in set:
        if i.val == point:
            return True
    return False


opened, closed = set(), set()
f_cost = Inf * ones(maze.shape[:], int)
opened.add(Node(entr))
f_cost[entr[1], entr[0]] = manhattan(entr, exit)
n = 0
while len(opened) > 0:
    n += 1
    current = choose_current(f_cost, list(opened), exit)
    opened.remove(current)
    closed.add(current)
    if current.val == exit:
        break
    for i in neighbours(maze, current.val):
        if check_closed(closed, i) is True:
            continue
        node = Node(i)
        node.next = current
        node.set_g(node.size() - 1)
        node.set_h(manhattan(i, exit))
        if node.get_f() < f_cost[i[1], i[0]]:
            f_cost[i[1], i[0]] = node.get_f()
            if check_node(opened, i) is True:
                if manhattan(i, entr) > node.get_g():
                    continue
            opened.add(node)

sol = ""
while current.next is not None:
    if current.val[1] == current.next.val[1] + 1:
        sol = "S" + sol
    elif current.val[1] == current.next.val[1] - 1:
        sol = "N" + sol
    elif current.val[0] == current.next.val[0] + 1:
        sol = "E" + sol
    elif current.val[0] == current.next.val[0] - 1:
        sol = "W" + sol
    current = current.next
print(n)
print(sol)

## Maze output

In [ ]:
txt = ""
with open(conf['OUTPUT_FILE'], 'w') as out:
    print(maze)
    for i in range(size[0]):
        for j in range(size[1]):
            out.write(base[maze[i][j]])
        out.write("\n")

    out.write("\n")
    out.write(f"{entr}          # entry (x,y)\n")
    out.write(f"{exit}          # exit (x,y)\n")
    out.write(sol)